# TP ClickHouse — Prise en main, CRUD et requêtage analytique

**Niveau :** Master Data / IA  
**Durée indicative :** 1h

## Objectifs

À l'issue de ce TP, vous devez être capables de :

- créer une base et une table dans ClickHouse ;
- insérer, consulter, modifier et supprimer des données ;
- utiliser les types et fonctions adaptés à l'analyse de données ;
- écrire des requêtes d'agrégation avec `GROUP BY`, `HAVING` et `ORDER BY` ;
- manipuler des dates ;
- utiliser des fonctions de fenêtrage ;
- comprendre quelques particularités de ClickHouse par rapport à une base relationnelle classique.

> **Important :** ClickHouse est principalement conçu pour l'analytique et les traitements OLAP. Les opérations de modification de données existent, mais leur comportement et leur coût diffèrent d'un SGBD OLTP comme PostgreSQL ou MySQL.


## 1. Connexion à ClickHouse

Le serveur ClickHouse doit être lancé avec le fichier `docker-compose-clickhouse.yml` fourni avec le TP.

Depuis le répertoire contenant ce fichier :

```bash
docker compose up -d clickhouse-01 clickhouse-02 clickhouse-03
```

Vérifier que le conteneur fonctionne :

```bash
docker ps
```

Vous pouvez ensuite utiliser l'interface HTTP de ClickHouse sur `http://localhost:8123`.

Pour ce TP, les requêtes SQL peuvent être exécutées depuis un notebook Python.


In [ ]:
# Installation du client Python si nécessaire
!pip install clickhouse-connect pandas


In [ ]:
import clickhouse_connect
import pandas as pd

client = clickhouse_connect.get_client(
    host="clickhouse-01",  # Remplacez par l'IP trouvée
    port=8123,
    username="clickhouseanf",
    password="anfnosql2026"
)

client.command("SELECT version()")


### Question 1

Quelle version de ClickHouse utilisez-vous ?

Complétez la cellule suivante.


In [ ]:
# Votre requête


## 2. Création de la base de données

Nous allons travailler sur une base consacrée à l'analyse de commandes d'une plateforme de commerce en ligne.

### Question 2

Créez une base de données appelée `ecommerce`.


In [ ]:
# Votre requête


### Question 3

Affichez les bases de données disponibles et vérifiez que `ecommerce` a bien été créée.


In [ ]:
# Votre requête


## 3. Création de la table `orders`

Nous allons utiliser la table suivante :

| Colonne | Type | Description |
|---|---|---|
| `order_id` | UInt32 | Identifiant de la commande |
| `customer_id` | UInt32 | Identifiant du client |
| `order_date` | Date | Date de commande |
| `product_category` | String | Catégorie du produit |
| `product` | String | Produit |
| `quantity` | UInt16 | Quantité commandée |
| `unit_price` | Float64 | Prix unitaire |
| `country` | String | Pays du client |
| `status` | String | Statut de la commande |

Créez la table `ecommerce.orders`.

Pour le moteur, utilisez `MergeTree` avec `ORDER BY order_id`.


In [ ]:
# Votre requête


### Question 4

Expliquez le rôle de `ENGINE = MergeTree` et de `ORDER BY` dans cette table.


In [ ]:
# Réponse


## 4. Insertion des données

Insérez les données suivantes dans `ecommerce.orders`.

Vous pouvez exécuter la cellule telle quelle ou observer sa structure avant de l'exécuter.


In [ ]:
client.command("""
INSERT INTO ecommerce.orders
(order_id, customer_id, order_date, product_category, product,
 quantity, unit_price, country, status)
VALUES
(1, 101, '2026-01-05', 'Informatique', 'Clavier mécanique', 2, 89.90, 'France', 'livree'),
(2, 102, '2026-01-07', 'Audio', 'Casque Bluetooth', 1, 129.90, 'France', 'livree'),
(3, 103, '2026-01-08', 'Informatique', 'Souris sans fil', 3, 39.90, 'Belgique', 'livree'),
(4, 101, '2026-01-12', 'Ecran', 'Ecran 27 pouces', 1, 299.90, 'France', 'expediee'),
(5, 104, '2026-01-15', 'Audio', 'Enceinte Bluetooth', 2, 79.90, 'Suisse', 'livree'),
(6, 105, '2026-01-18', 'Informatique', 'SSD 1 To', 2, 109.90, 'France', 'annulee'),
(7, 106, '2026-01-22', 'Ecran', 'Ecran 24 pouces', 2, 189.90, 'Espagne', 'livree'),
(8, 102, '2026-02-02', 'Informatique', 'Webcam HD', 1, 69.90, 'France', 'livree'),
(9, 107, '2026-02-04', 'Audio', 'Casque Bluetooth', 2, 129.90, 'Allemagne', 'expediee'),
(10, 108, '2026-02-10', 'Informatique', 'Clavier mécanique', 1, 89.90, 'France', 'livree'),
(11, 103, '2026-02-14', 'Ecran', 'Ecran 27 pouces', 2, 299.90, 'Belgique', 'livree'),
(12, 109, '2026-02-18', 'Audio', 'Enceinte Bluetooth', 1, 79.90, 'Italie', 'livree'),
(13, 110, '2026-02-21', 'Informatique', 'SSD 1 To', 1, 109.90, 'France', 'expediee'),
(14, 101, '2026-03-01', 'Audio', 'Casque Bluetooth', 1, 129.90, 'France', 'livree'),
(15, 105, '2026-03-05', 'Ecran', 'Ecran 24 pouces', 1, 189.90, 'France', 'livree'),
(16, 111, '2026-03-09', 'Informatique', 'Souris sans fil', 4, 39.90, 'Portugal', 'livree'),
(17, 112, '2026-03-12', 'Audio', 'Casque Bluetooth', 1, 129.90, 'France', 'annulee'),
(18, 108, '2026-03-15', 'Informatique', 'Webcam HD', 2, 69.90, 'France', 'livree'),
(19, 113, '2026-03-18', 'Ecran', 'Ecran 27 pouces', 1, 299.90, 'Allemagne', 'expediee'),
(20, 114, '2026-03-20', 'Audio', 'Enceinte Bluetooth', 3, 79.90, 'Espagne', 'livree')
""")


### Question 5

Affichez toutes les commandes.


In [ ]:
# Votre requête


### Question 6

Affichez uniquement les commandes livrées.


In [ ]:
# Votre requête


### Question 7

Affichez les commandes françaises dont le montant total est supérieur à 150 €.

Le montant total d'une ligne de commande est :

`quantity × unit_price`


In [ ]:
# Votre requête


## 5. Lire et explorer les données avec Python

Le client Python permet de récupérer directement le résultat sous forme de DataFrame.


In [ ]:
df = client.query_df("""
SELECT
    order_id,
    customer_id,
    order_date,
    product,
    quantity,
    unit_price,
    quantity * unit_price AS total_amount,
    country,
    status
FROM ecommerce.orders
ORDER BY order_date
""")

df


### Question 8

Calculez :

1. le nombre total de commandes ;
2. le nombre de clients différents ;
3. le chiffre d'affaires théorique total, en incluant tous les statuts.


In [ ]:
# Votre requête


## 6. UPDATE — modifier des données

Dans ClickHouse, `ALTER TABLE ... UPDATE` permet de modifier des données existantes. Cette opération est différente d'un `UPDATE` traditionnel d'un SGBD OLTP.

### Question 9

La commande `order_id = 4` doit être considérée comme livrée.

Modifiez son statut.


In [ ]:
# Votre requête


### Question 10

Vérifiez que la commande 4 possède maintenant le statut `livree`.


In [ ]:
# Votre requête


## 7. DELETE — supprimer des données

ClickHouse propose notamment `ALTER TABLE ... DELETE WHERE` pour supprimer les lignes correspondant à une condition.

### Question 11

La commande `order_id = 6` est annulée et doit être supprimée de notre jeu de données.

Supprimez-la puis vérifiez le résultat.


In [ ]:
# Votre requête


## 8. Requêtes analytiques

À partir de maintenant, les questions demandent de construire des requêtes SQL plus complexes.

### Question 12 — Chiffre d'affaires par pays

Calculez le chiffre d'affaires correspondant uniquement aux commandes `livree`, regroupé par pays.

Triez le résultat par chiffre d'affaires décroissant.


In [ ]:
# Votre requête


### Question 13 — Chiffre d'affaires par catégorie

Pour chaque catégorie, calculez :

- le nombre de commandes ;
- la quantité totale vendue ;
- le chiffre d'affaires.

Ne prenez en compte que les commandes livrées.


In [ ]:
# Votre requête


### Question 14 — HAVING

Identifiez les pays dont le chiffre d'affaires des commandes livrées est supérieur à 500 €.


In [ ]:
# Votre requête


### Question 15 — Produit le plus vendu

Pour chaque produit, calculez la quantité totale vendue, uniquement pour les commandes livrées.

Affichez les produits par quantité décroissante.


In [ ]:
# Votre requête


### Question 16 — Évolution mensuelle

Calculez le chiffre d'affaires mensuel des commandes livrées.

Vous devrez utiliser une fonction de ClickHouse permettant d'extraire le mois d'une date.


In [ ]:
# Votre requête


### Question 17 — Panier moyen

Calculez le montant moyen d'une commande livrée.

Attention : le montant d'une commande correspond à `quantity * unit_price`.


In [ ]:
# Votre requête


## 9. Fonctions spécifiques à ClickHouse

ClickHouse possède de nombreuses fonctions adaptées à l'analyse de données.

### Question 18 — Top 3 des clients

Identifiez les trois clients ayant généré le plus de chiffre d'affaires avec leurs commandes livrées.

Affichez :

- `customer_id`
- chiffre d'affaires
- nombre de commandes


In [ ]:
# Votre requête


### Question 19 — Analyse par date

Pour chaque jour où au moins une commande a été livrée, affichez :

- la date ;
- le nombre de commandes ;
- le chiffre d'affaires.

Triez chronologiquement.


In [ ]:
# Votre requête


## 10. Fenêtrage

Nous souhaitons comparer le chiffre d'affaires de chaque produit à celui de la catégorie à laquelle il appartient.

### Question 20

Construisez une requête qui calcule, pour chaque produit :

- son chiffre d'affaires ;
- le chiffre d'affaires total de sa catégorie ;
- la part du produit dans sa catégorie en pourcentage.

Vous pouvez utiliser une fonction de fenêtrage telle que `SUM(...) OVER (...)`.


In [ ]:
# Votre requête


## 11. Challenge — analyse complète

### Question 21

Construisez une requête permettant d'obtenir, pour chaque pays :

- le nombre de commandes livrées ;
- le nombre de clients distincts ;
- la quantité totale vendue ;
- le chiffre d'affaires ;
- le panier moyen.

Classez les résultats par chiffre d'affaires décroissant.

**Contrainte :** essayez de réaliser cette analyse avec une seule requête SQL.


In [ ]:
# Votre requête


## 12. Challenge final — détecter les clients actifs

### Question 22

Un client est considéré comme **actif** s'il possède au moins deux commandes livrées.

Identifiez tous les clients actifs et affichez :

- leur identifiant ;
- leur nombre de commandes livrées ;
- leur chiffre d'affaires ;
- leur première date de commande livrée ;
- leur dernière date de commande livrée.

Triez les clients par chiffre d'affaires décroissant.


In [ ]:
# Votre requête


## 13. Pour aller plus loin

### Question 23 — comparaison avec un SGBD relationnel

Répondez brièvement aux questions suivantes :

1. Quelle différence voyez-vous entre `ALTER TABLE ... UPDATE` de ClickHouse et un `UPDATE` classique dans PostgreSQL ?
2. Pourquoi ClickHouse utilise-t-il un moteur comme `MergeTree` ?
3. Quel est l'intérêt de `ORDER BY` dans la définition d'une table ClickHouse ?
4. Dans quels types de charges de travail ClickHouse est-il particulièrement adapté ?
5. Pourquoi ClickHouse n'est-il pas nécessairement le meilleur choix pour une application transactionnelle classique ?

### Question 24 — exploration

Utilisez `DESCRIBE TABLE` et les commandes d'information disponibles dans ClickHouse pour explorer la structure de `ecommerce.orders`.

Identifiez notamment les types des colonnes et les informations relatives au moteur de stockage.


In [ ]:
# Votre requête


## 14. Nettoyage

À la fin du TP, vous pouvez arrêter les conteneurs avec :

```bash
docker compose -f docker-compose-clickhouse.yml down
```

Pour supprimer également les volumes et donc les données persistantes :

```bash
docker compose -f docker-compose-clickhouse.yml down -v
```

> **Attention :** `down -v` supprime les données stockées dans les volumes Docker associés à ce TP.
